# Day 3: Random Forest from Scratch

Goal: Build a Random Forest on top of our Decision Tree implementation, understanding bootstrap sampling, feature randomness, and OOB evaluation.

## Imports

In [14]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from decision_tree import DecisionTree

## Bootstrap Sampling

Each tree in the forest trains on a different dataset. We achieve this by
sampling **with replacement** from the original data — this is called
**bootstrap sampling**.

Because we sample with replacement, some samples appear multiple times,
and roughly **37% of samples are never picked** at all. Those left-out
samples are called the **Out-of-Bag (OOB)** set for that tree and will
be used later for free validation.

In [15]:
def bootstrap_sampling(X: pd.DataFrame, y: np.ndarray) -> tuple[pd.DataFrame, np.ndarray, list]:
    num_of_indices = len(X)

    # choose random rows' indices with replacement
    # with replacement means index can appear more than once 
    random_indices = np.random.choice(a=num_of_indices, size=num_of_indices, replace=True)

    # get elements at selected indices 
    bootstrapped_X = X.iloc[random_indices]
    bootstrapped_y = y[random_indices]

    # out of bag - rows that were not selected and will be used for validation for curr tree
    oob_indices = list()

    for index in range(num_of_indices):
        if index not in random_indices:
            oob_indices.append(index)

    return bootstrapped_X, bootstrapped_y, oob_indices

## Random Feature Subset

Bootstrap sampling alone is not enough to create a diverse forest. If all features are available at every node, trees trained on similar data will still produce nearly identical splits, often dominated by the strongest predictive feature.

To force diversity, the algorithm randomly selects a subset of features (specifically, $\sqrt{\text{total features}}$) to evaluate at **every single split**, rather than picking one subset for the entire tree. If a dominant feature is excluded at the root node, the tree is forced to split on a different pattern, learning unique information.

### Implementation in `decision_tree.py`
We integrated this logic directly into our `DecisionTree` class:
* **`random_subspace` flag:** The constructor now accepts a boolean flag to indicate if the tree is part of a Random Forest.
* **Dynamic sampling:** Inside the `_best_split` method, right before evaluating thresholds, the tree checks this flag. If active, it calls `_sample_random_features`.
* **Without replacement:** The sampling strictly uses `replace=False` to ensure no feature is evaluated twice for the same split.

## Building the Forest

The forest is simply **n_trees independent experiments**. Each iteration:
1. Bootstrap sample → different data
2. Train one DecisionTree on that data using only random features at every node

We store each `(tree, oob_indices)` tuple so we can use them
later for OOB scoring and prediction.

In [16]:
def build_forest(X: pd.DataFrame, y: np.ndarray, n_trees: int) -> list[tuple]:
    trees = []

    for _ in range(n_trees):
        # create new tree
        new_tree = DecisionTree()
        # sample random rows and get out of bag indices
        training_data, results, oob_indices = bootstrap_sampling(X, y)

        # build tree
        new_tree.fit(X=training_data, y=results)

        trees.append((new_tree, oob_indices))

    return trees

## OOB Score

Since each tree only trained on ~63% of the data, we can use the
remaining ~37% (OOB samples) as a free validation set.

For each sample we collect predictions **only from trees that did NOT
train on it**, then take a majority vote and compare to the true label.
This gives us a reliable accuracy estimate without needing a separate
validation split.

In [17]:
def oob_score(X: pd.DataFrame, y: np.ndarray, forest_trees: list[tuple]) -> float:
    oob_predictions = defaultdict(list)
    correct_predictions = 0

    # validate each tree on out_of_bag rows of our DataFrame
    for decision_tree, oob_indices in forest_trees:
        for index in oob_indices:
            oob_predictions[index].extend(decision_tree.predict(X.iloc[[index]]))

    for index, predictions in oob_predictions.items():
        predictions = np.array(predictions)
        # choose the major prediction
        oob_prediction = 1 if np.sum(predictions == 1) >= (len(predictions) / 2) else 0

        # if oob_prediction is equal to actual answer - increment correct predictions counter
        if oob_prediction == y[index]:
            correct_predictions += 1

    # return total accuracy rate
    return correct_predictions / len(oob_predictions)

## Prediction

To predict on new data, every tree votes on each sample using its own
feature subset. The final prediction is the **majority vote** across
all trees — this is what makes the forest more robust than any single tree.

In [18]:
def predict(X: pd.DataFrame, forest_trees: list[tuple]) -> list[int]:
    predictions = list()    # get prediction for every row via majority vote 

    for index in range(len(X)):
        curr_predictions = list()   # collect predictions from all decision trees 
        for decision_tree, _ in forest_trees:
            curr_predictions.extend(decision_tree.predict(X.iloc[[index]]))

        curr_predictions = np.array(curr_predictions)
        # apply majority vote rule
        final_prediction = 1 if np.sum(curr_predictions) >= (len(curr_predictions) / 2) else 0

        predictions.append(final_prediction)

    return predictions

## Testing on a Toy Dataset

We will verify our Random Forest by training 5 trees on our synthetic dataset. We will calculate the Out-Of-Bag (OOB) score to estimate its performance on unseen data, and then check its accuracy on the training set.

In [19]:
# create the dummy dataset
data = {
    'Age': [22, 25, 47, 35, 14, 50, 28, 19, 60, 38],
    'Sex': ['male', 'female', 'female', 'male', 'male', 'female', 'male', 'female', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 1, 1, 0, 1, 0, 1]
}
df_dummy = pd.DataFrame(data)

X_dummy = df_dummy[['Age', 'Sex']]
y_dummy = df_dummy['Survived'].to_numpy()

# build a small forest
forest = build_forest(X_dummy, y_dummy, n_trees=5)

# evaluate the forest
oob_accuracy = oob_score(X_dummy, y_dummy, forest)
predictions = predict(X_dummy, forest)

print("OOB Accuracy: ", oob_accuracy)
print("Actual:       ", list(y_dummy))
print("Predictions:  ", predictions)
print("Accuracy:     ", np.mean(predictions == y_dummy))

OOB Accuracy:  0.7777777777777778
Actual:        [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]
Predictions:   [0, 1, 1, 0, 0, 1, 0, 1, 0, 1]
Accuracy:      0.9
